In [93]:
import os
import logging
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [94]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

OpenRouter API Key exists and begins sk-or-v1


In [95]:
openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key,base_url=openrouter_url)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [96]:
xiaomi_flash_model = "xiaomi/mimo-v2-flash"
deepseek_chat_model = "deepseek/deepseek-chat"
deepseek_model_3_2 = "deepseek/deepseek-v3.2"
mistralai_devstral_2512 = "mistralai/devstral-2512"
mistralai_mixtral_8x7b_instruct = "mistralai/mixtral-8x7b-instruct"
gemma4 = "google/gemma-4-31b-it"
qwen_qwen3_6_plus_free = "qwen/qwen3.5-397b-a17b"



In [97]:
DB_FILE = "school.db"
#Global connection variable
conn = None

In [ ]:
system_message = """You are an AI assistant designed to help teachers analyze student performance for parent-teacher meetings.

Your responsibilities:
- Understand the teacher’s question accurately
- Determine what type of academic data is needed
- Use the correct available tool
- Analyze returned data carefully
- Provide concise, teacher-friendly insights

Available tools:
1. get_marks(student_id)
   - Use for general marks requests when no specific timeframe is mentioned

2. get_marks_history(student_id)
   - Use for trends, improvement, decline, historical performance, or comparison over time

3. get_latest_marks(student_id)
   - Use for current performance, latest exam, recent scores, or present weaknesses

4. get_attendance(student_id)
   - Use when attendance or consistency may be relevant

Rules for tool usage:
- If student data is required, you MUST call the correct tool before answering
- Never guess or assume marks, attendance, or trends
- You may call multiple tools when needed
- Always wait for tool results before final analysis
- If the user asks about:
  • "improvement", "trend", "history", "decline" → use get_marks_history
  • "latest", "current", "recent", "now" → use get_latest_marks
  • general marks → use get_marks

Analysis requirements:
After receiving tool results:
- Identify strengths (best-performing subjects)
- Identify weaknesses (lowest-performing subjects)
- Mention trend if historical data is available
- Mention attendance impact if attendance is low
- Be objective and factual

Response style:
- Keep responses short (4–6 lines)
- Use simple language suitable for teachers and parents
- Be specific, not generic
- Avoid exaggeration
- If data is incomplete, clearly mention missing information

For summary requests, ALWAYS include:
1. Strength
2. Weakness
3. Attendance
4. Improvement suggestion

Example:
“Rahul performs strongly in Maths but needs improvement in English grammar. His recent scores show steady Maths performance but weaker language consistency. Attendance is 75%, which may be affecting overall progress. Daily reading and grammar practice are recommended.”

Your goal:
Help teachers quickly understand student performance, identify academic needs, and communicate effectively with parents."""

In [99]:
import sqlite3
from sqlite3 import Error


def create_connection(db_file):
    """Create a database connection"""
    try:
        conn = sqlite3.connect(db_file)
        conn.execute("PRAGMA foreign_keys = ON;")  # Enforce foreign keys
        print(f"Database is created.")
        return conn
    except Error as e:
        print(f"Database connection error: {e}")
        return None


def create_tables(conn):
    """Create tables if they do not exist"""
    try:
        cursor = conn.cursor()
        print("Create table enter ...")

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS subjects (
            id INTEGER PRIMARY KEY,
            name TEXT UNIQUE NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS exam_types (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            max_marks INTEGER NOT NULL
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS exams (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            exam_type_id INTEGER NOT NULL,
            date TEXT,
            FOREIGN KEY (exam_type_id) REFERENCES exam_types(id)
        );
        """)

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS marks (
            id INTEGER PRIMARY KEY,
            student_id INTEGER NOT NULL,
            subject_id INTEGER NOT NULL,
            exam_id INTEGER NOT NULL,
            marks_obtained REAL NOT NULL,
            FOREIGN KEY (student_id) REFERENCES students(id),
            FOREIGN KEY (subject_id) REFERENCES subjects(id),
            FOREIGN KEY (exam_id) REFERENCES exams(id)
        );
        """)

        # Prevent duplicates when mock data is inserted multiple times.
        cursor.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS idx_marks_unique
        ON marks(student_id, subject_id, exam_id);
        """)

        conn.commit()
        print("Tables created successfully (if not already existing).")

    except Error as e:
        print(f"Error while creating tables: {e}")
        conn.rollback()


In [100]:
#insert mock data
import sqlite3
from sqlite3 import Error


def insert_mock_data(conn):
    """Insert mock data safely"""
    try:
        cursor = conn.cursor()

        # STUDENTS
        students = [
            (101, 'Rahul'),
            (102, 'Anita'),
            (103, 'Kumar')
        ]
        cursor.executemany("""
            INSERT OR IGNORE INTO students (id, name)
            VALUES (?, ?)
        """, students)

        # SUBJECTS
        subjects = [
            (1, 'Maths'),
            (2, 'Tamil'),
            (3, 'Science'),
            (4, 'Social'),
            (5, 'English'),
            (6, 'AI')
        ]
        cursor.executemany("""
            INSERT OR IGNORE INTO subjects (id, name)
            VALUES (?, ?)
        """, subjects)

        # EXAM TYPES
        exam_types = [
            (1, 'Unit Test', 20),
            (2, 'Midterm', 40),
            (3, 'Final Exam', 100)
        ]
        cursor.executemany("""
            INSERT OR IGNORE INTO exam_types (id, name, max_marks)
            VALUES (?, ?, ?)
        """, exam_types)

        # EXAMS
        exams = [
            (1, 'Unit Test 1', 1, '2024-01-10'),
            (2, 'Unit Test 2', 1, '2024-02-10'),
            (3, 'Unit Test 3', 1, '2024-03-10'),
            (4, 'Pre Midterm', 2, '2024-06-15'),
            (5, 'Post Midterm', 2, '2024-07-15'),
            (6, 'Half Yearly', 3, '2024-09-10'),
            (7, 'Annual Exam', 3, '2025-02-10')
        ]
        cursor.executemany("""
            INSERT OR IGNORE INTO exams (id, name, exam_type_id, date)
            VALUES (?, ?, ?, ?)
        """, exams)

        # MARKS
        marks = [
            # Rahul
            (101, 1, 1, 18), (101, 1, 2, 17), (101, 1, 3, 19), (101, 1, 6, 78),
            (101, 5, 1, 10), (101, 5, 2, 12), (101, 5, 3, 11), (101, 5, 6, 55),
            (101, 3, 1, 15), (101, 3, 2, 16), (101, 3, 3, 17), (101, 3, 6, 70),

            # Anita
            (102, 1, 1, 12), (102, 1, 2, 14), (102, 1, 3, 15), (102, 1, 6, 65),
            (102, 5, 1, 18), (102, 5, 2, 19), (102, 5, 3, 17), (102, 5, 6, 80),

            # Kumar
            (103, 1, 1, 8), (103, 1, 2, 10), (103, 1, 3, 9), (103, 1, 6, 50),
            (103, 5, 1, 6), (103, 5, 2, 7), (103, 5, 3, 8), (103, 5, 6, 45)
        ]
        cursor.executemany("""
            INSERT OR IGNORE INTO marks (student_id, subject_id, exam_id, marks_obtained)
            VALUES (?, ?, ?, ?)
        """, marks)

        conn.commit()
        print("Mock data inserted successfully.")

    except Error as e:
        print(f"Error inserting mock data: {e}")
        conn.rollback()

    except Exception as e:
        print(f"Unexpected error: {e}")
        conn.rollback()

In [101]:
def simple_test_tables(conn):
    """Simple validation of all tables using basic queries"""
    try:
        cursor = conn.cursor()

        tables = ['students', 'subjects', 'exam_types', 'exams', 'marks']

        for table in tables:
            print(f"\n--- Testing {table} ---")

            # 1. Check if table has data
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = cursor.fetchone()[0]
            print(f"Total records: {count}")

            # 2. Fetch first 3 rows
            cursor.execute(f"SELECT * FROM {table} LIMIT 3")
            rows = cursor.fetchall()

            if rows:
                print("Sample data:")
                for row in rows:
                    print(row)
            else:
                print("No data found")

        print("\n✅ Simple testing completed successfully")

    except Exception as e:
        print(f"Error during simple testing: {e}")

In [102]:


def initialize_database(db_file="school.db"):
    """Main method to initialize DB"""
    try:
        conn = create_connection(db_file)
        print("Connection is created.")
        if conn is not None:
            print("Connection is not None.")
            create_tables(conn)
            insert_mock_data(conn)
            # If the DB already existed (previous runs), remove any accidental duplicates.
            conn.execute("""
            DELETE FROM marks
            WHERE id NOT IN (
                SELECT MIN(id)
                FROM marks
                GROUP BY student_id, subject_id, exam_id
            );
            """)
            conn.commit()
            simple_test_tables(conn)
        else:
            print("Failed to create database connection.")

    except Exception as e:
        print(f"Unexpected error: {e}")

    finally:
        if conn:
            conn.close()
            print("Database connection closed.")

In [103]:
initialize_database(db_file="school.db")

Database is created.
Connection is created.
Connection is not None.
Create table enter ...
Tables created successfully (if not already existing).
Mock data inserted successfully.

--- Testing students ---
Total records: 3
Sample data:
(101, 'Rahul')
(102, 'Anita')
(103, 'Kumar')

--- Testing subjects ---
Total records: 6
Sample data:
(1, 'Maths')
(2, 'Tamil')
(3, 'Science')

--- Testing exam_types ---
Total records: 3
Sample data:
(1, 'Unit Test', 20)
(2, 'Midterm', 40)
(3, 'Final Exam', 100)

--- Testing exams ---
Total records: 7
Sample data:
(1, 'Unit Test 1', 1, '2024-01-10')
(2, 'Unit Test 2', 1, '2024-02-10')
(3, 'Unit Test 3', 1, '2024-03-10')

--- Testing marks ---
Total records: 28
Sample data:
(1, 101, 1, 1, 18.0)
(2, 101, 1, 2, 17.0)
(3, 101, 1, 3, 19.0)

✅ Simple testing completed successfully
Database connection closed.


In [104]:
from collections import defaultdict

def get_connection(db_file: str = DB_FILE):
    # Keep connections short-lived for notebook/tool calls.
    conn = create_connection(db_file)
    if conn is None:
        raise RuntimeError(f"Failed to connect to database: {db_file}")
    return conn


def get_marks_history(student_id):
    logging.info(f"Fetching marks history for student_id={student_id}")

    conn = get_connection()
    try:
        cursor = conn.cursor()

        query = """ SELECT DISTINCT
    s.name AS subject,
    e.name AS exam,
    et.name AS exam_type,
    m.marks_obtained,
    et.max_marks,
    e.date
FROM marks m
JOIN subjects s ON m.subject_id = s.id
JOIN exams e ON m.exam_id = e.id
JOIN exam_types et ON e.exam_type_id = et.id
WHERE m.student_id = ?
ORDER BY e.date ASC """

        cursor.execute(query, (student_id,))
        rows = cursor.fetchall()

        if not rows:
            logging.warning("No marks found")
            return {"error": "No marks found"}

        data = defaultdict(list)

        for subject, exam, exam_type, score, max_marks, date in rows:
            data[subject].append({
                "exam": exam,
                "exam_type": exam_type,
                "score": score,
                "max_marks": max_marks,
                "date": date,
            })

        logging.info("Marks history fetched successfully")

        return {
            "student_id": student_id,
            "marks_history": dict(data),
        }
    finally:
        conn.close()

In [105]:
get_marks_history(101)

Database is created.


{'student_id': 101,
 'marks_history': {'Maths': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 18.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 17.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 19.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 78.0,
    'max_marks': 100,
    'date': '2024-09-10'}],
  'Science': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 15.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 16.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 17.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 7

In [106]:
def get_latest_marks(student_id):
    logging.info(f"Fetching latest marks for student_id={student_id}")

    conn = get_connection()
    try:
        cursor = conn.cursor()

        # Latest exam per subject for this student.
        query = """
WITH ranked AS (
    SELECT
        s.name AS subject,
        e.name AS exam,
        et.name AS exam_type,
        m.marks_obtained,
        et.max_marks,
        e.date,
        ROW_NUMBER() OVER (
            PARTITION BY s.id
            ORDER BY e.date DESC, e.id DESC
        ) AS rn
    FROM marks m
    JOIN subjects s ON m.subject_id = s.id
    JOIN exams e ON m.exam_id = e.id
    JOIN exam_types et ON e.exam_type_id = et.id
    WHERE m.student_id = ?
)
SELECT subject, exam, exam_type, marks_obtained, max_marks, date
FROM ranked
WHERE rn = 1
ORDER BY subject ASC;
"""

        cursor.execute(query, (student_id,))
        rows = cursor.fetchall()
    finally:
        conn.close()

    if not rows:
        logging.warning("No latest marks found")
        return {"error": "No marks found"}

    data = {}

    for subject, exam, exam_type, score, max_marks, date in rows:
        data[subject] = {
            "exam": exam,
            "exam_type": exam_type,
            "score": score,
            "max_marks": max_marks,
            "date": date,
        }

    logging.info("Latest marks fetched successfully")

    return {
        "student_id": student_id,
        "latest_marks": data,
    }
    if not rows:
        logging.warning("No latest marks found")
        return {"error": "No marks found"}

    data = {}

    for subject, exam, exam_type, score, max_marks, date in rows:
        data[subject] = {
            "exam": exam,
            "exam_type": exam_type,
            "score": score,
            "max_marks": max_marks,
            "date": date
        }

    logging.info("Latest marks fetched successfully")

    return {
        "student_id": student_id,
        "latest_marks": data
    }

In [107]:
get_latest_marks(101)

Database is created.


{'student_id': 101,
 'latest_marks': {'English': {'exam': 'Half Yearly',
   'exam_type': 'Final Exam',
   'score': 55.0,
   'max_marks': 100,
   'date': '2024-09-10'},
  'Maths': {'exam': 'Half Yearly',
   'exam_type': 'Final Exam',
   'score': 78.0,
   'max_marks': 100,
   'date': '2024-09-10'},
  'Science': {'exam': 'Half Yearly',
   'exam_type': 'Final Exam',
   'score': 70.0,
   'max_marks': 100,
   'date': '2024-09-10'}}}

In [108]:
def get_marks(student_id):
    logging.info(f"Fallback get_marks called for student_id={student_id}")
    return get_marks_history(student_id)

In [109]:
get_marks(101)

Database is created.


{'student_id': 101,
 'marks_history': {'Maths': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 18.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 17.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 19.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 78.0,
    'max_marks': 100,
    'date': '2024-09-10'}],
  'Science': [{'exam': 'Unit Test 1',
    'exam_type': 'Unit Test',
    'score': 15.0,
    'max_marks': 20,
    'date': '2024-01-10'},
   {'exam': 'Unit Test 2',
    'exam_type': 'Unit Test',
    'score': 16.0,
    'max_marks': 20,
    'date': '2024-02-10'},
   {'exam': 'Unit Test 3',
    'exam_type': 'Unit Test',
    'score': 17.0,
    'max_marks': 20,
    'date': '2024-03-10'},
   {'exam': 'Half Yearly',
    'exam_type': 'Final Exam',
    'score': 7

In [110]:
[
  {
    "type": "function",
    "function": {
      "name": "get_marks_history",
      "description": "Use when user asks for trends, past performance, or improvement over time",
      "parameters": {
        "type": "object",
        "properties": {
          "student_id": {"type": "string"}
        },
        "required": ["student_id"],
        "additionalProperties": False
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_latest_marks",
      "description": "Use when user asks for latest performance, current marks, or most recent exam",
      "parameters": {
        "type": "object",
        "properties": {
          "student_id": {"type": "string"}
        },
        "required": ["student_id"],
        "additionalProperties": False
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_marks",
      "description": "Fallback tool for general marks queries",
      "parameters": {
        "type": "object",
        "properties": {
          "student_id": {"type": "string"}
        },
        "required": ["student_id"],
        "additionalProperties": False
      }
    }
  }
]



[{'type': 'function',
  'function': {'name': 'get_marks_history',
   'description': 'Use when user asks for trends, past performance, or improvement over time',
   'parameters': {'type': 'object',
    'properties': {'student_id': {'type': 'string'}},
    'required': ['student_id'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_latest_marks',
   'description': 'Use when user asks for latest performance, current marks, or most recent exam',
   'parameters': {'type': 'object',
    'properties': {'student_id': {'type': 'string'}},
    'required': ['student_id'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_marks',
   'description': 'Fallback tool for general marks queries',
   'parameters': {'type': 'object',
    'properties': {'student_id': {'type': 'string'}},
    'required': ['student_id'],
    'additionalProperties': False}}}]

In [111]:
# IMPORTANT:
# - `TOOLS_SCHEMA` (below) is sent to the model -> must be JSON-serializable.
# - Actual Python callables live in `TOOLS_MAP` and are executed locally.

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_marks_history",
            "description": "Use when user asks for trends, past performance, or improvement over time",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_latest_marks",
            "description": "Use when user asks for latest performance, current marks, or most recent exam",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_marks",
            "description": "Fallback tool for general marks queries",
            "parameters": {
                "type": "object",
                "properties": {"student_id": {"type": "string"}},
                "required": ["student_id"],
                "additionalProperties": False,
            },
        },
    },
]

print(TOOLS_SCHEMA)

[{'type': 'function', 'function': {'name': 'get_marks_history', 'description': 'Use when user asks for trends, past performance, or improvement over time', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'get_latest_marks', 'description': 'Use when user asks for latest performance, current marks, or most recent exam', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'get_marks', 'description': 'Fallback tool for general marks queries', 'parameters': {'type': 'object', 'properties': {'student_id': {'type': 'string'}}, 'required': ['student_id'], 'additionalProperties': False}}}]


In [112]:
TOOLS_MAP = {
    "get_latest_marks": get_latest_marks,
    "get_marks_history": get_marks_history,
    "get_marks": get_marks
}

In [113]:
import json

def handle_tool_calls(assistant_msg):
    """Convert an assistant tool request into tool result messages."""

    tool_calls = getattr(assistant_msg, "tool_calls", None) or []
    tool_messages = []
    print("chat 3",tool_calls)
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        if tool_name not in TOOLS_MAP:
            raise ValueError(f"Unknown tool: {tool_name}")
       
        arguments = json.loads(tool_call.function.arguments or "{}")
        result = TOOLS_MAP[tool_name](**arguments)
        print("chat 4",result)
        tool_messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, default=str),
            }
        )

    return tool_messages

In [114]:
import json

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openrouter.chat.completions.create(
        model=gemma4,
        messages=messages,
        tools=TOOLS_SCHEMA,
        tool_choice="auto"
    )
    print("call 1",response.choices[0].message)
    while True:
        assistant_msg = response.choices[0].message
        tool_calls = getattr(assistant_msg, "tool_calls", None)

        if tool_calls:
            # 1) add assistant tool-request message
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_msg.content,
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in tool_calls
                    ],
                }
            )
            print("Call 2",messages)
            # 2) execute tools via helper
            messages.extend(handle_tool_calls(assistant_msg))

            # 3) call model again with tool outputs
            response = openrouter.chat.completions.create(
                model=gemma4,
                messages=messages,
                tools=TOOLS_SCHEMA,
                tool_choice="auto"
            )
        else:
            return assistant_msg.content or ""

In [115]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


call 1 ChatCompletionMessage(content="Hello! I'm here to help you analyze student performance for your parent-teacher meetings. Please provide a student ID and let me know what information you need.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None)
call 1 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-a73b1d0a8398aa80', function=Function(arguments='{"student_id": "101"}', name='get_marks_history'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-8c76fe3f33c50ad2', function=Function(arguments='{"student_id": "101"}', name='get_attendance'), type='function', index=1)], reasoning=None)
Call 2 [{'role': 'system', 'content': 'You are an AI assistant designed to help teachers analyze student performance for parent-teacher meetings.\n\nYour responsibilitie

Traceback (most recent call last):
  File "/Users/vasanthakumarjagannathan/projects/llm_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vasanthakumarjagannathan/projects/llm_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vasanthakumarjagannathan/projects/llm_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vasanthakumarjagannathan/projects/llm_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_inp